# Gradient Flow ODE Simulator — Full Replication

**Objective:** Numerically verify bias collapse across many targets and many values of m.

**Model:** $f(x) = \sum_{j=1}^m a_j \sigma(x - b_j)$, $\sigma(z) = \max(0,z)$, $x \in [-1,1]$

**ODEs:**
$$\dot{a}_j = -\int_{-1}^{1} (f(x) - f^*(x))\,\sigma(x - b_j)\,dx$$
$$\dot{b}_j = a_j \int_{b_j}^{1} (f(x) - f^*(x))\,dx$$

Figures are saved to: `figures/Replication data/{target}/{m}/{T}/`  
Already-completed runs are **skipped automatically** — delete the folder to re-run.

In [89]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import os

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## Core Functions

In [90]:
N_QUAD = 400
X_QUAD = np.linspace(-1, 1, N_QUAD)
DX     = X_QUAD[1] - X_QUAD[0]

def relu(z):
    return np.maximum(0.0, z)

def network(x, a, b):
    """f(x) = sum_j a_j * relu(x - b_j).  x:(N,), a,b:(m,) -> (N,)"""
    return (a * relu(x[:, None] - b[None, :])).sum(axis=1)

def make_ode(m, f_star):
    """Build ODE RHS for scipy.integrate.solve_ivp. State: y = [a(m), b(m)]."""
    fstar_vals = f_star(X_QUAD)

    def ode(t, y):
        a, b = y[:m], y[m:]
        residual  = network(X_QUAD, a, b) - fstar_vals          # (N,)
        relu_mat  = relu(X_QUAD[:, None] - b[None, :])          # (N, m)
        da        = -(residual[:, None] * relu_mat).sum(0) * DX  # (m,)
        cum_right = np.cumsum(residual[::-1])[::-1] * DX         # (N,)
        idx       = np.searchsorted(X_QUAD, b).clip(0, N_QUAD-1)
        db        = a * cum_right[idx]                           # (m,)
        return np.concatenate([da, db])

    return ode

def count_clusters(biases, tol=0.02):
    sorted_b = np.sort(biases)
    return 1 + int(np.sum(np.diff(sorted_b) > tol))

def get_cluster_centers(biases, tol=0.02):
    sorted_b = np.sort(biases)
    centers, group = [], [sorted_b[0]]
    for i in range(1, len(sorted_b)):
        if sorted_b[i] - sorted_b[i-1] > tol:
            centers.append(np.mean(group))
            group = []
        group.append(sorted_b[i])
    centers.append(np.mean(group))
    return np.array(centers)

def count_inflection_points(f_star, n=2000):
    x   = np.linspace(-1, 1, n)
    d2y = np.gradient(np.gradient(f_star(x), x), x)
    return int(np.sum(np.diff(np.sign(d2y)) != 0))

## Target Functions & Sweep Parameters

In [ ]:
# ---------- Target functions ----------
# folder_name : (label for plots, function)
TARGETS = {
    'x2':               (r'$x^2$',                           lambda x: x**2),
    'sin2pi_05sin4pi':  (r'$\sin(2\pi x)+0.5\sin(4\pi x)$', lambda x: np.sin(2*np.pi*x) + 0.5*np.sin(4*np.pi*x)),
    'x3_minus_x':       (r'$x^3 - x$',                       lambda x: x**3 - x),
    'exp_x':            (r'$e^x$',                            lambda x: np.exp(x)),
    'sin6pi':           (r'$\sin(6\pi x)$',                  lambda x: np.sin(6*np.pi*x)),
}

# ---------- Sweep parameters ----------
M_VALUES = [100, 500, 1000, 1500]
T_VALUES = [80, 200, 400, 800, 1000]
N_SAVE   = 500    # snapshots saved per run (only affects plot smoothness)
SEED     = 42

# Base output directory (relative to this notebook)
FIG_BASE = os.path.join('..', 'figures', 'Replication data')

total = len(TARGETS) * len(M_VALUES) * len(T_VALUES)
print(f"Total runs planned: {total}")
print(f"Targets : {list(TARGETS.keys())}")
print(f"m values: {M_VALUES}")
print(f"T values: {T_VALUES}")

Total runs planned: 20
Targets : ['x2', 'sin2pi_05sin4pi', 'x3_minus_x', 'exp_x', 'sin6pi']
m values: [100, 500]
T values: [80, 200]


## Main Simulation Loop

Each run saves two figures to `figures/Replication data/{target}/m={m}/T={T}/`:
- `slide93_reproduction.png` — bias trajectories, final fit, loss curve  
- `clusters_vs_inflections.png` — cluster locations overlaid on target

Runs are **skipped if both files already exist**.

In [92]:
done = 0
skipped = 0

for target_key, (target_label, f_star) in TARGETS.items():
    n_inf = count_inflection_points(f_star)

    for m in M_VALUES:
        for T in T_VALUES:
            done += 1

            # --- Output folder & skip logic ---
            out_dir = os.path.join(FIG_BASE, target_key, f'm={m}', f'T={T}')
            f1 = os.path.join(out_dir, 'slide93_reproduction.png')
            f2 = os.path.join(out_dir, 'clusters_vs_inflections.png')
            if os.path.exists(f1) and os.path.exists(f2):
                skipped += 1
                print(f"[{done}/{total}] SKIP  target={target_key}, m={m}, T={T}")
                continue

            print(f"[{done}/{total}] RUN   target={target_key}, m={m}, T={T} ...", end=' ', flush=True)
            os.makedirs(out_dir, exist_ok=True)

            # --- Initial conditions ---
            np.random.seed(SEED)
            a0 = np.random.randn(m) * 0.01
            b0 = np.random.uniform(-1, 1, m)
            y0 = np.concatenate([a0, b0])

            # --- Integrate ---
            sol = solve_ivp(
                make_ode(m, f_star),
                t_span=(0, T),
                y0=y0,
                method='RK45',
                t_eval=np.linspace(0, T, N_SAVE),
                rtol=1e-4,
                atol=1e-6,
                max_step=max(0.1, T / 500),
            )

            # --- Loss over time ---
            fstar_vals = f_star(X_QUAD)
            losses = np.array([
                0.5 * np.trapezoid((network(X_QUAD, sol.y[:m, i], sol.y[m:, i]) - fstar_vals)**2, X_QUAD)
                for i in range(sol.y.shape[1])
            ])

            # --- Final state ---
            a_final = sol.y[:m, -1]
            b_final = sol.y[m:, -1]
            x_plot  = np.linspace(-1, 1, 500)
            f_final = network(x_plot, a_final, b_final)

            n_clusters      = count_clusters(b_final)
            cluster_centers = get_cluster_centers(b_final)

            # ============================================================
            # Figure 1: Bias trajectories | Final fit | Loss curve
            # ============================================================
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            fig.suptitle(f'target={target_label}, m={m}, T={T}', fontsize=12)

            # Sorted bias trajectories
            ax = axes[0]
            sorted_b_traj = np.sort(sol.y[m:, :], axis=0)
            for j in range(m):
                ax.plot(sol.t, sorted_b_traj[j], color='steelblue', alpha=min(0.3, 15/m), linewidth=0.5)
            ax.set_xlabel('Time'); ax.set_ylabel('Bias value')
            ax.set_title('Sorted Bias Trajectories'); ax.set_xlim([0, T])

            # Final fit vs target
            ax = axes[1]
            ax.plot(x_plot, f_star(x_plot), 'k--', lw=2, label=f'Target {target_label}')
            ax.plot(x_plot, f_final, 'r-', lw=2, label='Network $f$')
            ax.scatter(b_final, np.zeros_like(b_final), c='steelblue', s=8, zorder=5, label='Biases')
            ax.set_xlabel('$x$'); ax.set_title('Final Fit vs Target')
            ax.legend(fontsize=8); ax.set_xlim([-1, 1])

            # Loss on log scale
            ax = axes[2]
            ax.semilogy(sol.t, losses, color='darkorange', lw=1.5)
            ax.set_xlabel('Time'); ax.set_ylabel('MSE Loss')
            ax.set_title(f'Loss (log)  final={losses[-1]:.2e}')
            ax.set_xlim([0, T]); ax.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig(f1, bbox_inches='tight')
            plt.close()

            # ============================================================
            # Figure 2: Cluster locations vs inflection points
            # ============================================================
            x_fine    = np.linspace(-1, 1, 2000)
            d2f       = np.gradient(np.gradient(f_star(x_fine), x_fine), x_fine)
            infl_x    = x_fine[:-1][np.diff(np.sign(d2f)) != 0]

            fig, ax = plt.subplots(figsize=(10, 4))
            fig.suptitle(f'target={target_label}, m={m}, T={T}  |  clusters={n_clusters}, inflections={n_inf}', fontsize=11)
            ax.plot(x_fine, f_star(x_fine), 'k-', lw=2, label=f'Target {target_label}')
            ax.plot(x_plot, f_final, 'r-', lw=1.5, label='Final network')
            for cx in cluster_centers:
                ax.axvline(cx, color='steelblue', alpha=0.6, lw=0.8)
            for ix in infl_x:
                ax.axvline(ix, color='green', alpha=0.5, lw=1.0, linestyle='--')
            from matplotlib.lines import Line2D
            h, _ = ax.get_legend_handles_labels()
            h += [Line2D([0],[0], color='steelblue', lw=1.5, label=f'Bias clusters ({n_clusters})'),
                  Line2D([0],[0], color='green',     lw=1.5, ls='--', label=f'Inflection pts ({n_inf})')]
            ax.legend(handles=h, fontsize=9)
            ax.set_xlabel('$x$')
            ax.set_title('Cluster Locations vs Inflection Points')
            plt.tight_layout()
            plt.savefig(f2, bbox_inches='tight')
            plt.close()

            print(f"clusters={n_clusters}, inflections={n_inf}, loss={losses[-1]:.5f}")

print(f"\nFinished. {done - skipped} new runs, {skipped} skipped.")

[1/20] SKIP  target=x2, m=100, T=80
[2/20] SKIP  target=x2, m=100, T=200
[3/20] SKIP  target=x2, m=500, T=80
[4/20] RUN   target=x2, m=500, T=200 ... clusters=16, inflections=0, loss=0.00013
[5/20] RUN   target=sin2pi_05sin4pi, m=100, T=80 ... clusters=16, inflections=7, loss=0.03874
[6/20] RUN   target=sin2pi_05sin4pi, m=100, T=200 ... clusters=9, inflections=7, loss=0.02593
[7/20] RUN   target=sin2pi_05sin4pi, m=500, T=80 ... clusters=21, inflections=7, loss=0.01577
[8/20] RUN   target=sin2pi_05sin4pi, m=500, T=200 ... 

KeyboardInterrupt: 